<a href="https://colab.research.google.com/github/santiagodie/sudoku-vision-solver/blob/main/02_digit_recognition_and_solving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Digit recognition and solving

This notebook completes the pipeline started in [notebook 01](01_grid_detection.ipynb): it reads the digit in each filled cell, solves the puzzle and draws the solution back onto the original image.

Two approaches to digit recognition are compared:

1. A **Random Forest trained on handwritten digits**, reused from an earlier exercise. It fails on these printed boards, and section 2 measures why.
2. **Tesseract OCR** restricted to the digits 1–9, which reads every clue correctly.

The puzzle is then solved with [Peter Norvig's constraint-propagation solver](https://github.com/norvig/pytudes/blob/main/ipynb/Sudoku.ipynb).

## Setup

In [ ]:
# On Colab: clone the repository (for the data) and install Tesseract.
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("data"):
        !git clone -q https://github.com/santiagodie/sudoku-vision-solver.git
        %cd sudoku-vision-solver
    !apt-get install -y -qq tesseract-ocr > /dev/null
    !pip install -q pytesseract

In [ ]:
import glob, os, re, time
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pytesseract
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
%matplotlib inline

BOARDS = ["data/boards/sudoku1.png", "data/boards/sudoku2.png", "data/boards/sudoku3.png"]
print("Tesseract", pytesseract.get_tesseract_version())

## 1. Grid detection

The same steps as notebook 01, condensed: binarise, keep the 81 square contours and sort them row by row. `show_image` and `sort_contours` come from the course material.

In [ ]:
# Provided in the course material.
def show_image(image):
    if len(image.shape) == 3:
        plt.imshow(image[:, :, ::-1])
    else:
        plt.imshow(image, cmap="gray")
    plt.axis("off")
    plt.show()

# Provided in the course material.
def sort_contours(cnts):
    boundingBoxes = [cv2.boundingRect(c) for c in cnts]
    (cnts, boundingBoxes) = zip(*sorted(zip(cnts, boundingBoxes), key=lambda b: (b[1][1], b[1][0]), reverse=False))
    return cnts

def is_cell(c):
    area = cv2.contourArea(c)
    x, y, w, h = cv2.boundingRect(c)
    solidity = area / float(cv2.contourArea(cv2.convexHull(c)))
    return solidity > 0.9 and 0.8 < w / float(h) < 1.2 and 1000 < area < 15000

def detect_grid(image_path):
    """Return the image, its binarised version (white ink on black) and the 81 sorted cells."""
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)[1]
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    cells = [c for c in contours if is_cell(c)]
    if len(cells) != 81:
        raise ValueError(f"Expected 81 cells, found {len(cells)}")
    return image, thresh, sort_contours(cells)

def crop_cell(image, c, margin=5):
    x, y, w, h = cv2.boundingRect(c)
    return image[y + margin:y + h - margin, x + margin:x + w - margin]

def is_empty(cell_crop):
    # A few stray pixels are tolerated; a digit has far more than 20.
    return np.count_nonzero(cell_crop) < 20

def print_grid(board_string):
    for r in range(9):
        row = board_string[r * 9:(r + 1) * 9]
        print(" ".join(row[0:3]), "|", " ".join(row[3:6]), "|", " ".join(row[6:9]))
        if r in (2, 5):
            print("------+-------+------")

### Reference boards

To measure recognition, the three boards were transcribed by hand from the images (`.` = empty). Accuracy below is the fraction of the 86 clues read correctly.

In [ ]:
REFERENCE = {
    "data/boards/sudoku1.png": "...62.3...7........3.794.65.8....6....59.8.23..4..3...6.....5.7.924.7...7438...9.",
    "data/boards/sudoku2.png": ".84..3.65.37.6....5...2......674..3..2....67.7....9.2.6.3..58....52.7.46....8....",
    "data/boards/sudoku3.png": "...73..4.7....48.....2..........29.8.3...8.7...796.25.6.23............2..8....5.6",
}

def clue_accuracy(read_board):
    """Apply a board-reading function to every board and count correctly read clues."""
    correct = total = 0
    for path, expected in REFERENCE.items():
        predicted = read_board(path)
        clues = [(p, e) for p, e in zip(predicted, expected) if e != "."]
        correct += sum(p == e for p, e in clues)
        total += len(clues)
    return correct, total

## 2. First attempt: a Random Forest trained on handwritten digits

An earlier course exercise trained a Random Forest on a dataset of **2,560 handwritten digits** (256 per class) in MNIST format: 28×28 grayscale images, white digit on black. The original model was saved as a 28 MB pickle; here it is retrained with the same hyperparameters, which takes a few seconds and avoids loading a pickled model.

Each image is described by its raw pixels, flattened into a 784-value vector (the `RawImage` descriptor from the course material).

In [ ]:
# Loading adapted from the course material (CargaYDescribe + RawImage descriptor).
def load_digits(folder="data/digits"):
    data, labels = [], []
    for path in sorted(glob.glob(f"{folder}/*/*.jpg")):
        data.append(cv2.imread(path, cv2.IMREAD_GRAYSCALE).flatten())
        labels.append(os.path.basename(os.path.dirname(path)))
    return np.array(data), np.array(labels)

data, labels = load_digits()
print("Dataset:", data.shape)

samples = [cv2.imread(sorted(glob.glob(f"data/digits/{d}/*.jpg"))[0], cv2.IMREAD_GRAYSCALE) for d in range(10)]
show_image(np.hstack(samples))

In [ ]:
trainData, testData, trainLabels, testLabels = train_test_split(
    data, labels, test_size=0.25, random_state=84, stratify=labels)

# Hyperparameters of the model from the earlier exercise.
start = time.time()
model = RandomForestClassifier(n_estimators=406, max_depth=30, min_samples_split=4,
                               max_features="sqrt", random_state=42, n_jobs=-1)
model.fit(trainData, trainLabels)
print(f"Trained in {time.time() - start:.1f} s\n")
print(classification_report(testLabels, model.predict(testData), digits=3))

Around 92% accuracy on handwritten digits. The first version applied it directly to the board, resizing each whole cell to 28×28:

In [ ]:
def read_board_rf(image_path, prepare):
    _, thresh, cells = detect_grid(image_path)
    board = ""
    for c in cells:
        crop = crop_cell(thresh, c)
        board += "." if is_empty(crop) else model.predict(prepare(crop).reshape(1, -1))[0]
    return board

def whole_cell(crop):
    return cv2.resize(crop, (28, 28), interpolation=cv2.INTER_AREA)

print_grid(read_board_rf(BOARDS[0], whole_cell))
correct, total = clue_accuracy(lambda p: read_board_rf(p, whole_cell))
print(f"\nWhole-cell input: {correct}/{total} clues correct ({correct / total:.0%})")

Almost everything is read as 1 or 9. Printed digits differ from handwritten ones, but there is a second, more mechanical difference: **framing**. MNIST digits are cropped to their bounding box, scaled to fit a 20×20 box and centred in the 28×28 image. The board cells were instead resized whole, so the digit fills only a small part of the input and is thinner than the strokes the model was trained on:

In [ ]:
def mnist_style(crop):
    """Crop to the digit, scale its longest side to 20 px and centre it on a 28x28 canvas, as in MNIST."""
    ys, xs = np.nonzero(crop)
    digit = crop[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = digit.shape
    scale = 20 / max(h, w)
    digit = cv2.resize(digit, (max(1, round(w * scale)), max(1, round(h * scale))), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((28, 28), np.uint8)
    h, w = digit.shape
    y0, x0 = (28 - h) // 2, (28 - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = digit
    return canvas

_, thresh, cells = detect_grid(BOARDS[0])
crop = crop_cell(thresh, cells[3])  # the 6 in the first row
training_six = cv2.imread(sorted(glob.glob("data/digits/6/*.jpg"))[0], cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
for ax, img, title in zip(axes, [training_six, whole_cell(crop), mnist_style(crop)],
                          ["Training sample", "Whole cell", "MNIST-style"]):
    ax.imshow(img, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()

In [ ]:
correct, total = clue_accuracy(lambda p: read_board_rf(p, mnist_style))
print(f"MNIST-style input: {correct}/{total} clues correct ({correct / total:.0%})")

Matching the framing helps a lot, but the model is still wrong on most clues: the remaining gap is the difference between handwriting and a printed typeface. Fixing it properly would mean training on printed digits, so instead the next section uses an OCR engine that is already built for printed text.

## 3. Tesseract OCR

Each cell is prepared for Tesseract, which works best on dark text over a white background with some margin:

1. Invert the crop, so the digit is black on white.
2. Upscale it to 84×84 px.
3. Smooth it with a 5×5 Gaussian blur.
4. Add a 20 px white border.

Tesseract runs with `--psm 6` (a uniform block of text) and a whitelist of the digits 1–9.

**Version note.** This configuration read every clue with Tesseract 4.1.1, the version on Colab. With Tesseract 5 it returns an empty string for two of the 9s. An empty result is dangerous here: the board string becomes shorter than 81 characters and every following cell shifts by one. So `read_digit` retries without the blur when the first pass does not return exactly one digit, and raises an error if that also fails.

In [ ]:
TESSERACT_CONFIG = "--psm 6 --oem 3 -c tessedit_char_whitelist=123456789"

def prepare_for_ocr(crop, blur=True):
    img = cv2.bitwise_not(crop)
    img = cv2.resize(img, (84, 84), interpolation=cv2.INTER_CUBIC)
    if blur:
        img = cv2.GaussianBlur(img, (5, 5), 0)
    return cv2.copyMakeBorder(img, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)

ocr_retries = 0

def read_digit(crop):
    global ocr_retries
    for blur in (True, False):
        text = pytesseract.image_to_string(prepare_for_ocr(crop, blur), config=TESSERACT_CONFIG).strip()
        if len(text) == 1:
            return text
        ocr_retries += 1
    raise ValueError(f"Could not read a single digit (last OCR output: {text!r})")

def read_board(image_path):
    _, thresh, cells = detect_grid(image_path)
    return "".join("." if is_empty(crop_cell(thresh, c)) else read_digit(crop_cell(thresh, c))
                   for c in cells)

In [ ]:
_, thresh, cells = detect_grid(BOARDS[0])
show_image(prepare_for_ocr(crop_cell(thresh, cells[3])))

start = time.time()
correct, total = clue_accuracy(read_board)
print(f"Tesseract: {correct}/{total} clues correct ({correct / total:.0%}) "
      f"in {time.time() - start:.1f} s, {ocr_retries} retries without blur")

In [ ]:
for path in BOARDS:
    board = read_board(path)
    print(path, "matches the reference:", board == REFERENCE[path])
    print_grid(board)
    print()

## 4. Solving

The solver is Peter Norvig's, from [pytudes](https://github.com/norvig/pytudes/blob/main/ipynb/Sudoku.ipynb) (MIT License, Copyright (c) 2010-2017 Peter Norvig), copied unchanged except for removing functions that are not used here. It represents the board as a dictionary from square (`'A1'` to `'I9'`) to the string of digits still possible there, and combines two techniques:

- **Constraint propagation.** When a square has a single possible digit, that digit is removed from all its peers. When a unit (row, column or box) has only one square left for a digit, the digit goes there.
- **Depth-first search.** When propagation gets stuck, it tries each possible digit of the square with the fewest options and propagates again.

The input format is the same `.` and digits string produced above.

In [ ]:
# --- Peter Norvig's Sudoku solver (pytudes, MIT License) ---
from typing import Dict, Optional

def cross(A, B) -> tuple:
    "Cross product of strings in A and strings in B."
    return tuple(a + b for a in A for b in B)

Digit     = str  # e.g. '1'
digits    = '123456789'
DigitSet  = str  # e.g. '123'
rows      = 'ABCDEFGHI'
cols      = digits
Square    = str  # e.g. 'A9'
squares   = cross(rows, cols)
Grid      = Dict[Square, DigitSet] # E.g. {'A9': '123', ...}
all_boxes = [cross(rs, cs)  for rs in ('ABC','DEF','GHI') for cs in ('123','456','789')]
all_units = [cross(rows, c) for c in cols] + [cross(r, cols) for r in rows] + all_boxes
units     = {s: tuple(u for u in all_units if s in u) for s in squares}
peers     = {s: set().union(*units[s]) - {s} for s in squares}
Picture   = str

def is_solution(solution: Grid, puzzle: Grid) -> bool:
    "Is this proposed solution to the puzzle actually valid?"
    return (solution is not None and
            all(solution[s] in puzzle[s] for s in squares) and
            all({solution[s] for s in unit} == set(digits) for unit in all_units))

def constrain(grid) -> Grid:
    "Propagate constraints on a copy of grid to yield a new constrained Grid."
    result: Grid = {s: digits for s in squares}
    for s in grid:
        if len(grid[s]) == 1:
            fill(result, s,  grid[s])
    return result

def fill(grid: Grid, s: Square, d: Digit) -> Optional[Grid]:
    """Eliminate all the digits except d from grid[s]."""
    if grid[s] == d or all(eliminate(grid, s, d2) for d2 in grid[s] if d2 != d):
        return grid
    else:
        return None

def eliminate(grid: Grid, s: Square, d: Digit) -> Optional[Grid]:
    """Eliminate d from grid[s]; implement the two constraint propagation strategies."""
    if d not in grid[s]:
        return grid        ## Already eliminated
    grid[s] = grid[s].replace(d, '')
    if not grid[s]:
        return None        ## None: no legal digit left
    elif len(grid[s]) == 1:
        # 1. If a square has only one possible digit, then eliminate that digit as a possibility for each of the square's peers.
        d2 = grid[s]
        if not all(eliminate(grid, s2, d2) for s2 in peers[s]):
            return None    ## None: can't eliminate d2 from some square
    for u in units[s]:
        dplaces = [s for s in u if d in grid[s]]
        # 2. If a unit has only one possible square that can hold a digit, then fill the square with the digit.
        if not dplaces or (len(dplaces) == 1 and not fill(grid, dplaces[0], d)):
            return None    ## None: no place in u for d
    return grid

def parse(picture) -> Grid:
    """Convert a Picture to a Grid."""
    vals = re.findall(r"[.1-9]|[{][1-9]+[}]", picture)
    assert len(vals) == 81
    return {s: digits if v == '.' else re.sub(r"[{}]", '', v)
            for s, v in zip(squares, vals)}

def picture(grid) -> Picture:
    """Convert a Grid to a Picture string, one line at a time."""
    if grid is None:
        return "None"
    def val(d: DigitSet) -> str: return '.' if d == digits else d if len(d) == 1 else '{' + d + '}'
    maxwidth = max(len(val(grid[s])) for s in grid)
    dash1 = '-' * (maxwidth * 3 + 2)
    dash3 = '\n' + '+'.join(3 * [dash1])
    def cell(r, c): return val(grid[r + c]).center(maxwidth) + ('|'  if c in '36' else ' ')
    def line(r): return ''.join(cell(r, c) for c in cols)    + (dash3 if r in 'CF' else '')
    return '\n'.join(map(line, rows))

def search(grid) -> Grid:
    "Depth-first search with constraint propagation to find a solution."
    if grid is None:
        return None
    s = min((s for s in squares if len(grid[s]) > 1),
            default=None, key=lambda s: len(grid[s]))
    if s is None: # No squares with multiple possibilities; the search has succeeded
        return grid
    for d in grid[s]:
        solution = search(fill(grid.copy(), s, d))
        if solution:
            return solution
    return None

def print_side_by_side(left, right, width=20):
    """Print two strings side-by-side, line-by-line, each side `width` wide."""
    for L, R in zip(left.splitlines(), right.splitlines()):
        print(L.ljust(width), R.ljust(width))
# --- end of Norvig's code ---

A small wrapper solves a board string, verifies the result with Norvig's `is_solution` and returns the 81 solved digits as a string, in the same order as the cells.

In [ ]:
def solve(board_string):
    puzzle = parse(board_string)
    solution = search(constrain(puzzle))
    if not is_solution(solution, puzzle):
        raise ValueError("No valid solution: the board was probably misread")
    return "".join(solution[s] for s in squares)

for path in BOARDS:
    board = read_board(path)
    start = time.perf_counter()
    solution = solve(board)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"{path}: solved in {elapsed:.1f} ms")
    print_side_by_side("Puzzle:\n" + picture(parse(board)), "Solution:\n" + picture(parse(solution)))
    print()

## 5. Drawing the solution

The missing digits are written in red into the empty cells, reusing the cell contours from the detection step to position them.

In [ ]:
def draw_solution(image, cells, board, solution):
    result = image.copy()
    for c, given, digit in zip(cells, board, solution):
        if given == ".":
            x, y, w, h = cv2.boundingRect(c)
            cv2.putText(result, digit, (x + w // 3, y + (2 * h) // 3),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA)
    return result

def solve_image(image_path):
    """Full pipeline: detect the grid, read the clues, solve and draw the solution."""
    image, thresh, cells = detect_grid(image_path)
    board = "".join("." if is_empty(crop_cell(thresh, c)) else read_digit(crop_cell(thresh, c))
                    for c in cells)
    return draw_solution(image, cells, board, solve(board))

In [ ]:
for path in BOARDS:
    show_image(solve_image(path))